In [2]:

# Import the required libraries
import pandas as pd
import numpy as np
import pickle

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder, OneHotEncoder

# Load the dataset
data = pd.read_csv('Churn_Modelling.csv')

# Check the dataset
print("Dataset shape:", data.shape)
print("\nColumns:")
print(data.columns.tolist())

# Display the first five rows
data.head()


Dataset shape: (10000, 14)

Columns:
['RowNumber', 'CustomerId', 'Surname', 'CreditScore', 'Geography', 'Gender', 'Age', 'Tenure', 'Balance', 'NumOfProducts', 'HasCrCard', 'IsActiveMember', 'EstimatedSalary', 'Exited']


,RowNumber,CustomerId,Surname,CreditScore,Geography,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Exited
0,1,15634602,Hargrave,619,France,Female,42,2,0.00,1,1,1,101348.88,1
1,2,15647311,Hill,608,Spain,Female,41,1,83807.86,1,0,1,112542.58,0
2,3,15619304,Onio,502,France,Female,42,8,159660.80,3,1,0,113931.57,1
3,4,15701354,Boni,699,France,Female,39,1,0.00,2,0,0,93826.63,0
4,5,15737888,Mitchell,850,Spain,Female,43,2,125510.82,1,1,1,79084.10,0


In [3]:

# Check categorical columns and missing values
print("Categorical columns:")
print(data.select_dtypes(include=['object']).columns.tolist())

print("\nMissing values:")
print(data.isnull().sum())


Categorical columns:
['Surname', 'Geography', 'Gender']

Missing values:
RowNumber          0
CustomerId         0
Surname            0
CreditScore        0
Geography          0
Gender             0
Age                0
Tenure             0
Balance            0
NumOfProducts      0
HasCrCard          0
IsActiveMember     0
EstimatedSalary    0
Exited             0
dtype: int64


C:\Users\chauh\AppData\Local\Temp\ipykernel_9544\508882005.py:3: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  print(data.select_dtypes(include=['object']).columns.tolist())


In [4]:

# Drop columns that are not useful for prediction
data = data.drop(['RowNumber', 'CustomerId', 'Surname'], axis=1)

# Encode Gender: Male/Female into numerical values
label_encoder_gender = LabelEncoder()
data['Gender'] = label_encoder_gender.fit_transform(data['Gender'])

# One-hot encode Geography
onehot_encoder_geo = OneHotEncoder(handle_unknown='ignore', sparse_output=False)

geo_encoded = onehot_encoder_geo.fit_transform(data[['Geography']])

# Convert encoded Geography into a DataFrame
geo_encoded_df = pd.DataFrame(
    geo_encoded,
    columns=onehot_encoder_geo.get_feature_names_out(['Geography']),
    index=data.index
)

# Replace Geography with its encoded columns
data = pd.concat(
    [data.drop('Geography', axis=1), geo_encoded_df],
    axis=1
)

# Check the processed data
print(data.head())
print("\nColumns:", data.columns.tolist())


   CreditScore  Gender  Age  Tenure    Balance  NumOfProducts  HasCrCard  \
0          619       0   42       2       0.00              1          1   
1          608       0   41       1   83807.86              1          0   
2          502       0   42       8  159660.80              3          1   
3          699       0   39       1       0.00              2          0   
4          850       0   43       2  125510.82              1          1   

   IsActiveMember  EstimatedSalary  Exited  Geography_France  \
0               1        101348.88       1               1.0   
1               1        112542.58       0               0.0   
2               0        113931.57       1               1.0   
3               0         93826.63       0               1.0   
4               1         79084.10       0               0.0   

   Geography_Germany  Geography_Spain  
0                0.0              0.0  
1                0.0              1.0  
2                0.0              0.0 

In [19]:
X = data.drop(['EstimatedSalary', 'Exited'], axis=1)
y = data['EstimatedSalary']

print("X columns:", X.columns.tolist())
print("X shape:", X.shape)
print("y shape:", y.shape)


X columns: ['CreditScore', 'Gender', 'Age', 'Tenure', 'Balance', 'NumOfProducts', 'HasCrCard', 'IsActiveMember', 'Geography_France', 'Geography_Germany', 'Geography_Spain']
X shape: (10000, 11)
y shape: (10000,)


In [20]:

# Split the dataset into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42
)

# Check the sizes
print("X_train:", X_train.shape)
print("X_test:", X_test.shape)
print("y_train:", y_train.shape)
print("y_test:", y_test.shape)


X_train: (8000, 11)
X_test: (2000, 11)
y_train: (8000,)
y_test: (2000,)


In [21]:

# Create the scaler
scaler = StandardScaler()

# Fit the scaler on training data and transform it
X_train = scaler.fit_transform(X_train)

# Use the same scaler to transform testing data
X_test = scaler.transform(X_test)

# Check the shapes after scaling
print("X_train shape:", X_train.shape)
print("X_test shape:", X_test.shape)


X_train shape: (8000, 11)
X_test shape: (2000, 11)


In [22]:

# Save the regression scaler separately
with open('regression_scaler.pkl', 'wb') as file:
    pickle.dump(scaler, file)

print("Regression scaler saved successfully!")


Regression scaler saved successfully!


In [23]:

# Import ANN components
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense

# Build the regression model
model = Sequential([
    # First hidden layer: 64 neurons
    Dense(64, activation='relu', input_shape=(X_train.shape[1],)),

    # Second hidden layer: 32 neurons
    Dense(32, activation='relu'),

    # Output layer: 1 neuron for salary prediction
    Dense(1)
])

# Display the model architecture
model.summary()


c:\Users\chauh\OneDrive\Desktop\ANN_CHRUN_APP\.venv\Lib\site-packages\keras\src\layers\core\dense.py:107: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense_3 (Dense)                 │ (None, 64)             │           768 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_4 (Dense)                 │ (None, 32)             │         2,080 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_5 (Dense)                 │ (None, 1)              │            33 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 2,881 (11.25 KB)

 Trainable params: 2,881 (11.25 KB)

 Non-trainable params: 0 (0.00 B)

In [24]:

# Compile the regression model
model.compile(
    optimizer='adam',
    loss='mean_absolute_error',
    metrics=['mae']
)


In [25]:

from tensorflow.keras.callbacks import EarlyStopping, TensorBoard
from datetime import datetime

# Create a unique folder for TensorBoard logs
log_dir = "logs/regression/" + datetime.now().strftime("%Y%m%d-%H%M%S")

# Monitor validation loss and stop if it stops improving
early_stopping = EarlyStopping(
    monitor='val_loss',
    patience=10,
    restore_best_weights=True
)

# Record training metrics for TensorBoard
tensorboard_callback = TensorBoard(
    log_dir=log_dir,
    histogram_freq=1
)

# Train the ANN
history = model.fit(
    X_train,
    y_train,
    validation_split=0.2,
    epochs=100,
    batch_size=32,
    callbacks=[early_stopping, tensorboard_callback]
)


Epoch 1/100
200/200 ━━━━━━━━━━━━━━━━━━━━ 2s 4ms/step - loss: 99970.0312 - mae: 99970.0312 - val_loss: 102005.7188 - val_mae: 102005.7188
Epoch 2/100
200/200 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - loss: 99539.1719 - mae: 99539.1719 - val_loss: 101136.2969 - val_mae: 101136.2969
Epoch 3/100
200/200 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - loss: 97995.2891 - mae: 97995.2891 - val_loss: 98825.5625 - val_mae: 98825.5625
Epoch 4/100
200/200 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - loss: 94790.9688 - mae: 94790.9688 - val_loss: 94688.4922 - val_mae: 94688.4922
Epoch 5/100
200/200 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - loss: 89800.4688 - mae: 89800.4688 - val_loss: 88779.1172 - val_mae: 88779.1172
Epoch 6/100
200/200 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - loss: 83245.3984 - mae: 83245.3984 - val_loss: 81479.8906 - val_mae: 81479.8906
Epoch 7/100
200/200 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - loss: 75789.4453 - mae: 75789.4453 - val_loss: 73738.2422 - val_mae: 73738.2422
Epoch 8/100
200/200 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/ste

In [26]:
test_loss, test_mae = model.evaluate(X_test, y_test, verbose=0)

print(f"Test MAE: {test_mae:.2f}")

Test MAE: 50439.84


In [27]:
y_pred = model.predict(X_test).flatten()

results = pd.DataFrame({
    'Actual Salary': y_test.to_numpy(),
    'Predicted Salary': y_pred
})

print(results.head(10))

63/63 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step  
   Actual Salary  Predicted Salary
0       41788.37      82964.843750
1      146379.30      94614.703125
2       58561.31     103173.484375
3      170679.74      95579.156250
4      114669.79     102280.070312
5      149418.41      89696.117188
6       75685.97     104787.156250
7       70529.00     109706.929688
8       16618.76      99632.695312
9      164104.74      98711.140625


In [28]:
from sklearn.metrics import mean_absolute_error

baseline_pred = [y_train.mean()] * len(y_test)

baseline_mae = mean_absolute_error(y_test, baseline_pred)

print(f"Baseline MAE: {baseline_mae:.2f}")
print(f"ANN MAE: {test_mae:.2f}")

Baseline MAE: 49781.35
ANN MAE: 50439.84


In [29]:
model.save('ann_regression_salary_model.keras')

print("Regression model saved successfully!")

Regression model saved successfully!


In [30]:
import pickle

with open('regression_scaler.pkl', 'wb') as file:
    pickle.dump(scaler, file)

with open('regression_label_encoder_gender.pkl', 'wb') as file:
    pickle.dump(label_encoder_gender, file)

with open('regression_onehot_encoder_geo.pkl', 'wb') as file:
    pickle.dump(onehot_encoder_geo, file)

print("Regression preprocessing objects saved!")

Regression preprocessing objects saved!


In [31]:
print(scaler.feature_names_in_)

['CreditScore' 'Gender' 'Age' 'Tenure' 'Balance' 'NumOfProducts'
 'HasCrCard' 'IsActiveMember' 'Geography_France' 'Geography_Germany'
 'Geography_Spain']
